In [1]:
total_cards = 52

red_cards = 26

hearts = 13

face_cards = 12

diamond_face = 3

spade_face = 3

queens = 4

overlap = 1

p_red = red_cards / total_cards
print("P(Red Card):", p_red)

p_heart_given_red = hearts / red_cards
print("P(Heart | Red):", p_heart_given_red)

p_diamond_given_face = diamond_face / face_cards
print("P(Diamond | Face Card):", p_diamond_given_face)

p_spade_or_queen = (spade_face + queens - overlap) / face_cards
print("P(Spade OR Queen | Face Card):", p_spade_or_queen)

P(Red Card): 0.5
P(Heart | Red): 0.5
P(Diamond | Face Card): 0.25
P(Spade OR Queen | Face Card): 0.5


In [2]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([
    ('Intelligence', 'Grade'),
    ('StudyHours', 'Grade'),
    ('Difficulty', 'Grade'),
    ('Grade', 'Pass')
])

cpd_i = TabularCPD('Intelligence', 2, [[0.3], [0.7]])
cpd_s = TabularCPD('StudyHours', 2, [[0.4], [0.6]])
cpd_d = TabularCPD('Difficulty', 2, [[0.4], [0.6]])

cpd_g = TabularCPD(
    variable='Grade', variable_card=3,
    values=[
        [0.05, 0.10, 0.15, 0.20, 0.25, 0.35, 0.40, 0.50],
        [0.25, 0.30, 0.35, 0.40, 0.45, 0.40, 0.40, 0.35],
        [0.70, 0.60, 0.50, 0.40, 0.30, 0.25, 0.20, 0.15]
    ],
    evidence=['Intelligence', 'StudyHours', 'Difficulty'],
    evidence_card=[2, 2, 2]
)

cpd_p = TabularCPD(
    variable='Pass', variable_card=2,
    values=[
        [0.05, 0.20, 0.50],
        [0.95, 0.80, 0.50]
    ],
    evidence=['Grade'],
    evidence_card=[3]
)

model.add_cpds(cpd_i, cpd_s, cpd_d, cpd_g, cpd_p)
assert model.check_model()

infer = VariableElimination(model)

r1 = infer.query(['Pass'], evidence={'StudyHours': 1, 'Difficulty': 0})
print("\n=== Task 2 ===")
print("P(Pass | StudyHours=Sufficient, Difficulty=Hard):")
print(r1)

r2 = infer.query(['Intelligence'], evidence={'Pass': 1})
print("P(Intelligence | Pass=Yes):")
print(r2)

ModuleNotFoundError: No module named 'pgmpy'

In [ ]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([
    ('Disease', 'Fever'),
    ('Disease', 'Cough'),
    ('Disease', 'Fatigue'),
    ('Disease', 'Chills')
])

cpd_D = TabularCPD(variable='Disease', variable_card=2,
                   values=[[0.3], [0.7]])

# Fever
cpd_Fever = TabularCPD(
    variable='Fever', variable_card=2,
    values=[[0.9, 0.5], [0.1, 0.5]],
    evidence=['Disease'],
    evidence_card=[2]
)

cpd_Cough = TabularCPD(
    variable='Cough', variable_card=2,
    values=[[0.8, 0.6], [0.2, 0.4]],
    evidence=['Disease'],
    evidence_card=[2]
)

cpd_Fatigue = TabularCPD(
    variable='Fatigue', variable_card=2,
    values=[[0.7, 0.3], [0.3, 0.7]],
    evidence=['Disease'],
    evidence_card=[2]
)

cpd_Chills = TabularCPD(
    variable='Chills', variable_card=2,
    values=[[0.6, 0.4], [0.4, 0.6]],
    evidence=['Disease'],
    evidence_card=[2]
)

model.add_cpds(cpd_D, cpd_Fever, cpd_Cough, cpd_Fatigue, cpd_Chills)

print("Model valid:", model.check_model())

inference = VariableElimination(model)

q1 = inference.query(variables=['Disease'],
                     evidence={'Fever': 0, 'Cough': 0})
print("\nP(Disease | Fever, Cough):")
print(q1)

q2 = inference.query(variables=['Disease'],
                     evidence={'Fever': 0, 'Cough': 0, 'Chills': 0})
print("\nP(Disease | Fever, Cough, Chills):")
print(q2)

q3 = inference.query(variables=['Fatigue'],
                     evidence={'Disease': 0})
print("\nP(Fatigue | Flu):")

In [ ]:
import numpy as np

states = ["Sunny", "Cloudy", "Rainy"]

transition = np.array([
    [0.6, 0.3, 0.1],  # Sunny
    [0.3, 0.4, 0.3],  # Cloudy
    [0.2, 0.3, 0.5]   # Rainy
])

current = 0
days = []

for i in range(10):
    current = np.random.choice([0,1,2], p=transition[current])
    days.append(states[current])

print("Weather for 10 days:")
print(days)

rainy_days = days.count("Rainy")
print("Rainy days:", rainy_days)

count = 0
trials = 10000

for _ in range(trials):
    current = 0
    rainy = 0
    for i in range(10):
        current = np.random.choice([0,1,2], p=transition[current])
        if states[current] == "Rainy":
            rainy += 1
    if rainy >= 3:
        count += 1

print("P(at least 3 rainy days):", count / trials)